# Titanic Modeling & Classification Pipeline (Part B)

Imputation and scaling happen **after** the train-test split, inside scikit-learn `Pipeline` / `ColumnTransformer` objects, so transformers never see test rows (no leakage).

## B.1 Load Titanic CSV and stratified split

Load the committed offline fallback `titanic.csv`. Do **not** call `sns.load_dataset` here.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

os.makedirs("figures", exist_ok=True)

# Load from committed fallback (DO NOT use sns.load_dataset here)
df = pd.read_csv("titanic.csv")

features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X, y = df[features], df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set shape: {X_train.shape}, Test set shape: {X_test.shape}")
print("Training class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))

Training set shape: (712, 7), Test set shape: (179, 7)
Training class balance:
 survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64
Test class balance:
 survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64


## B.2 Leak-free column transformer

Numeric median impute + `StandardScaler`; categorical most-frequent impute + one-hot. Both are nested in the model pipeline so they fit only on training folds.

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["age", "sibsp", "parch", "fare", "pclass"]
cat = ["sex", "embarked"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

## B.3 Train three classifiers and plot the decision tree

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

trained_pipes = {}
for name, model in models.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
    trained_pipes[name] = pipe
    print(f"Fitted {name}")

# Plot Decision Tree with feature names
tree_clf = trained_pipes["Decision Tree"].named_steps["clf"]
feature_names = trained_pipes["Decision Tree"].named_steps["pre"].get_feature_names_out()

plt.figure(figsize=(20, 10))
plot_tree(tree_clf, feature_names=feature_names, class_names=["Not Survived", "Survived"], filled=True, rounded=True)
plt.title("Decision Tree Visualization (Max Depth = 4)")
plt.savefig("figures/decision_tree.png")
plt.show()

Fitted Logistic Regression
Fitted Decision Tree


Fitted Random Forest


C:\Users\advik\AppData\Local\Temp\ipykernel_8108\3775116294.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## B.4 Evaluate all three classifiers and plot ROC curves

In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import numpy as np

eval_results = []
plt.figure(figsize=(8, 6))

for name, pipe in trained_pipes.items():
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    eval_results.append({"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC": auc})

    print(f"\n{name} confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison")
plt.legend()
plt.savefig("figures/roc_curves.png")
plt.show()

metrics_df = pd.DataFrame(eval_results)
print(metrics_df)


Logistic Regression confusion matrix:
[[98 12]
 [23 46]]

Decision Tree confusion matrix:
[[103   7]
 [ 31  38]]

Random Forest confusion matrix:
[[96 14]
 [21 48]]


                 Model  Accuracy  Precision    Recall        F1       AUC
0  Logistic Regression  0.804469   0.793103  0.666667  0.724409  0.843742
1        Decision Tree  0.787709   0.844444  0.550725  0.666667  0.821014
2        Random Forest  0.804469   0.774194  0.695652  0.732824  0.830698


C:\Users\advik\AppData\Local\Temp\ipykernel_8108\4177962715.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## B.5 Imbalance comparison (SMOTE vs class weight)

`imblearn.pipeline.Pipeline` applies SMOTE **after** preprocessing and only on the training data passed to `.fit`.

In [5]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# SMOTE Pipeline ensures resampling happens inside cross-validation/training only
smote_pipe = ImbPipeline([
    ("pre", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(n_estimators=200, random_state=42))
])

balanced_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42))
])

smote_pipe.fit(X_train, y_train)
balanced_pipe.fit(X_train, y_train)

print("SMOTE F1:", f1_score(y_test, smote_pipe.predict(X_test)))
print("Class Weight Balanced F1:", f1_score(y_test, balanced_pipe.predict(X_test)))

SMOTE F1: 0.746268656716418
Class Weight Balanced F1: 0.75


## B.6 GridSearchCV with out-of-bag (OOB) score

Tune Random Forest with 5-fold CV on F1. `oob_score=True` is computed on the best estimator after the search refits on the full training set.

In [6]:
from sklearn.model_selection import GridSearchCV

rf_pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(oob_score=True, random_state=42))
])

grid = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [4, 6, 8, None],
    "clf__max_features": ["sqrt", "log2"]
}

gs = GridSearchCV(rf_pipe, grid, cv=5, scoring="f1", n_jobs=-1)
gs.fit(X_train, y_train)

print("Best Parameters:", gs.best_params_)
print("Best CV F1:", gs.best_score_)
print("OOB Score:", gs.best_estimator_.named_steps["clf"].oob_score_)
print("Held-out F1:", f1_score(y_test, gs.predict(X_test)))

Best Parameters: {'clf__max_depth': 4, 'clf__max_features': 'sqrt', 'clf__n_estimators': 100}
Best CV F1: 0.7483092675554793
OOB Score: 0.8146067415730337
Held-out F1: 0.6949152542372882


## B.7 Regression side-task (predicting fare)

`fare` is the target, so it is **not** passed through the classifier preprocessor. A separate transformer uses only `[pclass, sex, age, sibsp, parch, embarked]`.

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

reg_features = ["pclass", "sex", "age", "sibsp", "parch", "embarked"]
X_reg, y_reg = df[reg_features], df["fare"]

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg_num = ["age", "sibsp", "parch", "pclass"]
reg_cat = ["sex", "embarked"]
reg_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), reg_num),
    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), reg_cat)
])

reg_pipe = Pipeline([
    ("pre", reg_preprocessor),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42))
])

reg_pipe.fit(X_reg_train, y_reg_train)
y_pred_reg = reg_pipe.predict(X_reg_test)

mae = mean_absolute_error(y_reg_test, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_reg))
r2 = r2_score(y_reg_test, y_pred_reg)
n = len(y_reg_test)
p = X_reg_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print(f"MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f} | Adj R2: {adj_r2:.2f}")

# Residual plot
residuals = y_reg_test - y_pred_reg
plt.figure(figsize=(8, 5))
plt.scatter(y_pred_reg, residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Fare")
plt.ylabel("Residuals")
plt.title("Fare Regression Residual Plot (Heteroscedasticity check)")
plt.savefig("figures/fare_residuals.png")
plt.show()

MAE: 15.40 | RMSE: 41.07 | R2: -0.09 | Adj R2: -0.13


C:\Users\advik\AppData\Local\Temp\ipykernel_8108\3567311504.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## B.8 & B.9 Save full pipeline and verify re-loading

In [8]:
import joblib

best_pipeline = gs.best_estimator_
joblib.dump(best_pipeline, "titanic_pipeline.joblib")

# Reload and test prediction
loaded_pipe = joblib.load("titanic_pipeline.joblib")
sample_passenger = pd.DataFrame([{
    "pclass": 3, "sex": "male", "age": 22, "sibsp": 1, "parch": 0, "fare": 7.25, "embarked": "S"
}])

print("Prediction:", loaded_pipe.predict(sample_passenger))
print("Probability:", loaded_pipe.predict_proba(sample_passenger))
print("Same as in-memory pipeline:", np.allclose(
    loaded_pipe.predict_proba(sample_passenger),
    best_pipeline.predict_proba(sample_passenger)
))

Prediction: [0]
Probability: [[0.88978595 0.11021405]]
Same as in-memory pipeline: True
